# Assignment 1

## Question 1

### Data Preparation

In [22]:
!pip install pandas
!pip install numpy

In [23]:
import pandas as pd

# Daily adjusted close prices for S&P 500 constituents, 1970–2025.
prices = pd.read_csv("data/prices.csv", parse_dates=[0])

# Daily closing values for the S&P 500 index (ˆGSPC), same date range.
market = pd.read_csv("data/market.csv", parse_dates=[0])

# Daily 10-year U.S. Treasury constant maturity rate from FRED, 1962–2025.
# This is an annualized yield expressed as a percentage.
dgs10 = pd.read_csv("data/DGS10.csv", parse_dates=[0])


# Keep only data from the last 10 years.
def filter_last_10_years(df: pd.DataFrame) -> pd.DataFrame:
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    cutoff = df[date_col].max() - pd.DateOffset(years=10)
    return df.loc[df[date_col] >= cutoff].reset_index(drop=True)


prices_10y = filter_last_10_years(prices)
market_10y = filter_last_10_years(market)
dgs10_10y = filter_last_10_years(dgs10)

prices_10y.head(), market_10y.head(), dgs10_10y.head()

(        Date          A       AAPL       ABBV  ABNB        ABT       ACGL  \
 0 2015-03-04  38.679150  28.706537  39.771614   NaN  38.854362  18.811941   
 1 2015-03-05  38.881775  28.230852  37.521389   NaN  39.209774  18.989443   
 2 2015-03-06  38.246311  28.273281  36.716324   NaN  38.432858  18.976765   
 3 2015-03-09  38.439720  28.393879  36.650341   NaN  38.705601  19.179623   
 4 2015-03-10  37.417477  27.806528  36.848297   NaN  38.160099  18.954576   
 
          ACN       ADBE        ADI  ...         WTW         WY        WYNN  \
 0  77.067551  77.629997  47.521145  ...  107.532234  23.238997  121.084457   
 1  78.033951  78.620003  47.561821  ...  108.143448  23.144056  121.241493   
 2  76.864090  77.550003  47.138695  ...  106.875664  22.398127  119.496986   
 3  76.965813  77.930000  47.569962  ...  107.622734  22.743965  115.868446   
 4  74.651497  76.010002  46.544689  ...  106.649300  22.540529  112.963898   
 
          XEL        XOM        XYL        YUM        

In [24]:
# Filter out stocks not present on both the first and last day
# of the 10-year span.
#
# The downside of this strategy is that it creates a survivorship bias
# because it only includes stocks that are still in the S&P 500 after 10 years.
# This means that the portfolio is biased towards stocks that have performed well
# over the last 10 years, which may not be representative of the overall market.
#
# The upside of this strategy is it allows us to easily calculate the covariance
# matrix (for question 1(b)). Every pair of two stocks will have their covariance
# calculated over the same 10-year period. As a result, the resulting matrix is
# more likely to be "positive semi-definite," preventing errors in later tasks
# that require matrix inversion or optimization.

# Prepare data for filtering
date_col = prices_10y.columns[0]
prices_sorted = prices_10y.sort_values(date_col).reset_index(drop=True)
price_cols = prices_sorted.columns.drop(date_col)

first_row = prices_sorted.iloc[0]
last_row = prices_sorted.iloc[-1]

present_first = first_row[price_cols].notna()
present_last = last_row[price_cols].notna()

# Keep only stocks present on both the first and last day of the 10-year span.
keep_cols = price_cols[present_first & present_last]
removed_cols = price_cols.difference(keep_cols).tolist()

prices_10y_filtered = prices_sorted[[date_col] + list(keep_cols)]

print(f"Stocks before filtering: {len(price_cols)}")
print(f"Stocks after filtering:  {len(keep_cols)}")
print("Removed stocks:")
print(removed_cols)

Stocks before filtering: 503
Stocks after filtering:  472
Removed stocks:
['ABNB', 'CARR', 'CEG', 'CRWD', 'CTVA', 'DAY', 'DELL', 'DOW', 'FOX', 'FOXA', 'FTV', 'GDDY', 'GEHC', 'GEV', 'HPE', 'HWM', 'INVH', 'IR', 'KHC', 'KVUE', 'LW', 'MRNA', 'OTIS', 'PLTR', 'PYPL', 'SOLV', 'SW', 'UBER', 'VICI', 'VLTO', 'VST']


In [25]:
# Verify there are no missing values in the remaining stocks that would require
# forward filling. In this case, there are no such missing values.
date_col_filtered = prices_10y_filtered.columns[0]
price_cols_filtered = prices_10y_filtered.columns.drop(date_col_filtered)

missing_counts = prices_10y_filtered[price_cols_filtered].isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

print(f"Stocks with missing values: {len(missing_counts)}")
print(missing_counts)

# Preprocessing for prices.csv and market.csv is complete.

Stocks with missing values: 0
Series([], dtype: int64)


### Task (a)


In [26]:
import numpy as np

# Compute daily log returns for each stock
stock_log_returns = np.log1p(prices_10y_filtered[price_cols_filtered].pct_change())
stock_returns = pd.concat(
    [prices_10y_filtered[[date_col_filtered]], stock_log_returns], axis=1
)

# Compute daily log returns for the market index (^GSPC)
market_date_col = market_10y.columns[0]
market_col = market_10y.columns.drop(market_date_col)[0]
market_log_returns = np.log1p(market_10y[[market_col]].pct_change())
market_returns = pd.concat([market_10y[[market_date_col]], market_log_returns], axis=1)

# Combine individual stock and market log returns into one dataframe.
log_returns = stock_returns.merge(
    market_returns,
    left_on=date_col_filtered,
    right_on=market_date_col,
    # Keep only rows where both stock and market have data.
    how="inner",
)

if market_date_col != date_col_filtered:
    print(f"Dropping {market_date_col} column")
    log_returns = log_returns.drop(columns=[market_date_col])

# The first row is NaN because returns use a one-day lag (t-1). Drop it.
return_cols = log_returns.columns.drop(date_col_filtered)
log_returns = log_returns.dropna(subset=return_cols).reset_index(drop=True)

log_returns.head()

,Date,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS,^GSPC
0,2015-03-05,0.005225,-0.016709,-0.058242,0.009106,0.009391,0.012462,0.012672,0.000856,0.008460,...,-0.004094,0.001296,0.010071,-0.005060,-0.003087,0.005479,0.003530,0.001427,0.004963,0.001195
1,2015-03-06,-0.016479,0.001502,-0.021690,-0.020013,-0.000668,-0.015105,-0.013703,-0.008936,-0.026247,...,-0.032761,-0.014493,-0.037333,-0.012880,-0.011590,-0.017034,-0.024032,-0.016926,-0.014744,-0.014275
2,2015-03-09,0.005044,0.004256,-0.001799,0.007072,0.010633,0.001323,0.004888,0.009107,0.003022,...,0.015323,-0.030836,0.012698,-0.005504,0.002272,0.003531,0.005144,-0.002569,0.011079,0.003937
3,2015-03-10,-0.026953,-0.020903,0.005387,-0.014194,-0.011803,-0.030531,-0.024946,-0.021789,-0.018932,...,-0.008985,-0.025387,0.002345,-0.010624,-0.015726,-0.019833,-0.012389,-0.018059,-0.012828,-0.017107
4,2015-03-11,0.005400,-0.018400,0.013872,0.002811,0.015926,-0.013721,0.000000,0.001572,-0.000439,...,0.004503,-0.022489,-0.009117,-0.002853,-0.003464,-0.017878,0.006214,0.002275,0.006761,-0.001920


### Task (b)

In [27]:
# Align risk-free rate (DGS10) to log_returns dates to ensure
# we are working with the same set of dates.
rf_date_col = dgs10_10y.columns[0]
rf_aligned = log_returns[[date_col_filtered]].merge(
    dgs10_10y,
    left_on=date_col_filtered,
    right_on=rf_date_col,
    how="inner",
)

if rf_date_col != date_col_filtered:
    rf_aligned = rf_aligned.drop(columns=[rf_date_col])

rf_aligned.head()

,Date,DGS10
0,2016-02-11,1.63
1,2016-02-12,1.74
2,2016-02-16,1.78
3,2016-02-17,1.81
4,2016-02-18,1.75


In [28]:
# Check missing values exist in DGS10 (aligned risk-free rate).
missing_dgs10 = rf_aligned["DGS10"].isna().sum()
print(f"Missing DGS10 values: {missing_dgs10}")

# Forward fill missing DGS10 values.
# Forward filling is appropriate here because it avoids look-ahead bias
# that would occur from linear interpolation or backfiling.
rf_aligned["DGS10"] = rf_aligned["DGS10"].ffill()

print(f"Missing DGS10 values after ffill: {rf_aligned['DGS10'].isna().sum()}")

Missing DGS10 values: 16
Missing DGS10 values after ffill: 0


In [29]:
# Convert annualized DGS10 (%) to daily log risk-free rate using the given formula:
# Daily risk-free rate = ln(1 + y/100) / 252
rf_aligned["rf_daily"] = np.log(1 + rf_aligned["DGS10"] / 100) / 252

rf_aligned[[date_col_filtered, "DGS10", "rf_daily"]].head()

,Date,DGS10,rf_daily
0,2016-02-11,1.63,0.000064
1,2016-02-12,1.74,0.000068
2,2016-02-16,1.78,0.000070
3,2016-02-17,1.81,0.000071
4,2016-02-18,1.75,0.000069


In [30]:
# Calculate the average daily risk-free rate (r_f) across the entire period.
mean_rf_daily = rf_aligned["rf_daily"].mean()

# Isolate only the stock return columns for matrix operations.
returns_only = log_returns[price_cols_filtered]

# 1) Compute Annualized Expected Excess Returns (mu_hat)
# Formula: mu_hat = 252 * mean_daily_returns - annualized_rf
mean_daily_returns = returns_only.mean()
# Annualize the risk-free rate.
annualized_rf = mean_rf_daily * 252
mu_hat = 252 * mean_daily_returns - annualized_rf

# 2) Compute Annualized Covariance Matrix (sigma_hat)
# Formula: sigma_hat = 252 * sigma_daily
sigma_daily = returns_only.cov()
sigma_hat = sigma_daily * 252

# Reporting
print(f"mu_hat shape: {mu_hat.shape}")
print(f"sigma_hat shape: {sigma_hat.shape}")
print("mu_hat (first 5):\n", mu_hat.head())
print("sigma_hat (top-left 5x5):\n", sigma_hat.iloc[:5, :5])

mu_hat shape: (472,)
sigma_hat shape: (472, 472)
mu_hat (first 5):
 A       0.090744
AAPL    0.185837
ABBV    0.140429
ABT     0.101467
ACGL    0.133680
dtype: float64
sigma_hat (top-left 5x5):
              A      AAPL      ABBV       ABT      ACGL
A     0.071053  0.035979  0.025718  0.035982  0.026281
AAPL  0.035979  0.080883  0.021141  0.030205  0.025429
ABBV  0.025718  0.021141  0.071062  0.027341  0.020596
ABT   0.035982  0.030205  0.027341  0.056182  0.023860
ACGL  0.026281  0.025429  0.020596  0.023860  0.072284


### Task (c)

In [31]:
# Annualized volatility (sqrt of diagonal of sigma_hat)
annualized_vol = pd.Series(np.sqrt(np.diag(sigma_hat)), index=sigma_hat.index)

# Find the top 2 and bottom 2 stocks, measured by annualized excess returns.
top2_returns = mu_hat.sort_values(ascending=False).head(2)
bottom2_returns = mu_hat.sort_values(ascending=True).head(2)

# Find the top 2 stocks, measured by annualized volatility.
top2_vol = annualized_vol.sort_values(ascending=False).head(2)

# Market correlation (with ^GSPC)
market_col = market_10y.columns.drop(market_10y.columns[0])[0]
market_corr = log_returns[price_cols_filtered].corrwith(log_returns[market_col])
# Find the stock with the lowest correlation with the market.
lowest_corr = market_corr.sort_values(ascending=True).head(1)

print("Top 2 annualized excess returns:")
print(top2_returns)
print("Bottom 2 annualized excess returns:")
print(bottom2_returns)
print("Top 2 annualized volatility:")
print(top2_vol)
print("Lowest correlation with market:")
print(lowest_corr)

# Summary table for outliers
outlier_tickers = pd.Index([])
outlier_tickers = outlier_tickers.union(top2_returns.index)
outlier_tickers = outlier_tickers.union(bottom2_returns.index)
outlier_tickers = outlier_tickers.union(top2_vol.index)
outlier_tickers = outlier_tickers.union(lowest_corr.index)

anomaly_summary = pd.DataFrame(
    {
        "Ticker": outlier_tickers,
        "Annualized Return": mu_hat.loc[outlier_tickers].values,
        "Annualized Volatility": annualized_vol.loc[outlier_tickers].values,
    }
).reset_index(drop=True)

anomaly_summary

Top 2 annualized excess returns:
NVDA    0.512736
AMD     0.325222
dtype: float64
Bottom 2 annualized excess returns:
WBA    -0.191229
VTRS   -0.188427
dtype: float64
Top 2 annualized volatility:
ENPH    0.803022
SMCI    0.673020
dtype: float64
Lowest correlation with market:
NEM    0.19653
dtype: float64


,Ticker,Annualized Return,Annualized Volatility
0,AMD,0.325222,0.574632
1,ENPH,0.119787,0.803022
2,NEM,0.050907,0.360399
3,NVDA,0.512736,0.489970
4,SMCI,0.205238,0.673020
5,VTRS,-0.188427,0.386344
6,WBA,-0.191229,0.352821


# Question 3

In [32]:
mu_hat

A       0.090744
AAPL    0.185837
ABBV    0.140429
ABT     0.101467
ACGL    0.133680
          ...   
XYL     0.114830
YUM     0.095085
ZBH    -0.027762
ZBRA    0.093576
ZTS     0.112578
Length: 472, dtype: float64

In [33]:
sigma_hat

,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
A,0.071053,0.035979,0.025718,0.035982,0.026281,0.036982,0.043904,0.045105,0.025993,0.032048,...,0.026856,0.040906,0.044843,0.016422,0.023874,0.038501,0.025002,0.031326,0.051134,0.038631
AAPL,0.035979,0.080883,0.021141,0.030205,0.025429,0.038773,0.054523,0.051108,0.023709,0.033805,...,0.025740,0.040790,0.048278,0.017527,0.024382,0.034405,0.026253,0.027801,0.052695,0.035187
ABBV,0.025718,0.021141,0.071062,0.027341,0.020596,0.021728,0.024550,0.023402,0.018676,0.022857,...,0.019011,0.026355,0.027618,0.013887,0.021643,0.021130,0.017255,0.025242,0.025095,0.026888
ABT,0.035982,0.030205,0.027341,0.056182,0.023860,0.030700,0.035512,0.032457,0.021001,0.029614,...,0.024408,0.033366,0.026829,0.020347,0.017392,0.029861,0.022482,0.030415,0.035951,0.033752
ACGL,0.026281,0.025429,0.020596,0.023860,0.072284,0.030776,0.025875,0.031622,0.031043,0.033679,...,0.031568,0.043913,0.047059,0.022047,0.033230,0.036154,0.026797,0.031066,0.033097,0.025010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XYL,0.038501,0.034405,0.021130,0.029861,0.036154,0.036691,0.036641,0.044614,0.030851,0.035915,...,0.030497,0.048309,0.051621,0.019985,0.032135,0.071134,0.028594,0.032352,0.047799,0.031950
YUM,0.025002,0.026253,0.017255,0.022482,0.026797,0.027317,0.028707,0.030438,0.022160,0.027047,...,0.024113,0.037188,0.040908,0.017530,0.022804,0.028594,0.057256,0.027508,0.031888,0.026994
ZBH,0.031326,0.027801,0.025242,0.030415,0.031066,0.030950,0.030027,0.035130,0.026233,0.028522,...,0.025534,0.041126,0.052612,0.015687,0.029923,0.032352,0.027508,0.073542,0.036201,0.029175
ZBRA,0.051134,0.052695,0.025095,0.035951,0.033097,0.046082,0.056161,0.063717,0.033553,0.040328,...,0.031369,0.055493,0.067262,0.013898,0.033231,0.047799,0.031888,0.036201,0.159615,0.044707


In [34]:
sigma_inv = np.linalg.inv(sigma_hat)
ones = np.ones(len(mu_hat))

A = ones @ sigma_inv @ mu_hat
B = mu_hat @ sigma_inv @ mu_hat
C = ones @ sigma_inv @ ones
D = B * C - A**2

In [35]:
def mv_weights(mu_target):
    lam_1 = (C * mu_target - A) / D
    lam_2 = (B - A * mu_target) / D
    return sigma_inv @ (lam_1 * mu_hat + lam_2 * ones)

In [36]:
mv_weights(2)

array([ 1.51681383e-02,  5.35021388e-02,  1.64034840e-01,  9.41023843e-02,
       -5.95509409e-02,  6.30853430e-02, -1.09396936e-01,  3.27668020e-01,
       -7.22327791e-02, -3.28600707e-02, -2.36390495e-02, -1.58366245e-01,
        2.16252264e-01, -4.64048289e-02,  2.92736444e-01, -1.77934935e-01,
        2.87569716e-02,  2.09651999e-01, -6.49370984e-02,  1.28591996e-02,
       -9.78177006e-03, -1.05284342e-02, -3.19140836e-02, -1.83487291e-01,
        7.30555926e-02,  4.86637999e-03, -2.14700359e-01,  3.32767796e-02,
       -1.11761637e-01, -3.13818075e-02,  1.61684443e-02,  1.91236280e-03,
       -2.18659473e-01, -1.44933032e-01, -7.84206258e-02, -1.03582221e-01,
        1.37079018e-02, -1.56965262e-01,  8.65466132e-02, -1.83112360e-01,
       -3.43705239e-02,  3.98110178e-01, -2.27386886e-01,  1.54581409e-01,
        2.17145466e-01, -7.14178672e-02,  3.30748719e-02,  2.38467707e-02,
        8.67033193e-02, -8.15175508e-02,  9.26628271e-02, -6.93198956e-02,
       -3.35630292e-02, -

## GMV Portfolio

In [37]:
mu_gmv = A / C
sigma_gmv = np.sqrt(1 / C)

## Frontier

In [38]:
mu_grid = np.linspace(mu_gmv, mu_hat.values.max() * 1.2, 300)
sigma_grid = np.sqrt((C * mu_grid**2 - 2*A*mu_grid + B) / D)

In [39]:
asset_sigma = np.sqrt(np.diag(sigma_hat.values))
asset_mu = mu_hat.values
tickers = mu_hat.index.tolist()

In [40]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(sigma_grid, mu_grid, color="#003366", linewidth=2.2, label="Efficient Frontier")

ax.scatter(sigma_gmv, mu_gmv, color="#cc0000", zorder=5, s=80, label="GMV Portfolio")
ax.annotate("GMV", xy=(sigma_gmv, mu_gmv),
            xytext=(sigma_gmv + 0.005, mu_gmv - 0.01),
            fontsize=8, color="#cc0000")

ax.scatter(asset_sigma, asset_mu, color="#888888", zorder=4, s=40,
           marker="D", label="Individual Assets")
for i, t in enumerate(tickers):
    ax.annotate(t, xy=(asset_sigma[i], asset_mu[i]),
                xytext=(asset_sigma[i] + 0.002, asset_mu[i] + 0.003),
                fontsize=6, color="#444444")

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_xlabel("Annualized Volatility $\\sigma$", fontsize=12)
ax.set_ylabel("Annualized Excess Return $\\mu$", fontsize=12)
ax.set_title("Efficient Frontier (Closed-Form)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

# 3c)

## GMV Portfolio

In [ ]:
w_gmv  = sigma_inv @ ones / C
mu_gmv = w_gmv @ mu_hat.values
sigma_gmv = np.sqrt(w_gmv @ sigma_hat.values @ w_gmv)

print("GMV Portfolio")
print(f"Expected Return: {mu_gmv:.2%}")
print(f"Std Deviation: {sigma_gmv:.2%}")
print(f"Weights:")
gmv_weights = pd.Series(w_gmv, index=mu_hat.index).sort_values(ascending=False)
print(gmv_weights.to_string())
print(f"Long positions: {(w_gmv > 0).sum()}")
print(f"Short positions: {(w_gmv < 0).sum()}")

GMV Portfolio
Expected Return: 7.15%
Std Deviation: 8.14%
Weights:
ED       0.094311
JNJ      0.085993
VZ       0.085191
HBAN     0.077394
AMCR     0.068858
BK       0.066819
UNP      0.059810
KDP      0.058698
WMT      0.058695
BG       0.056689
HLT      0.056302
EVRG     0.054199
RSG      0.053667
CBOE     0.053536
ICE      0.046509
OMC      0.045989
LIN      0.045847
K        0.044786
LMT      0.044038
REG      0.043419
ZBH      0.042973
AZO      0.042714
NEM      0.040937
CPT      0.040636
CLX      0.040540
GOOG     0.040084
SNPS     0.039764
WTW      0.039020
EXPD     0.038948
SRE      0.038495
MCD      0.037475
PCAR     0.036876
PTC      0.035211
AIZ      0.034941
DIS      0.032849
YUM      0.032533
JPM      0.032332
KO       0.031892
BMY      0.031528
HUM      0.030994
MRK      0.030258
GWW      0.030133
APH      0.030120
O        0.029993
PSA      0.029783
BDX      0.029266
GRMN     0.028897
DVA      0.028754
AEP      0.028385
CMG      0.028168
PANW     0.028109
TJX      0.0280

## Tangency Portfolio

In [42]:
w_tan_raw = sigma_inv @ mu_hat.values
w_tan = w_tan_raw / (ones @ w_tan_raw)

mu_tan = w_tan @ mu_hat.values
sigma_tan = np.sqrt(w_tan @ sigma_hat.values @ w_tan)
sharpe_tan = mu_tan / sigma_tan

In [43]:
print("\nTangency Portfolio")
print(f"Expected Return: {mu_tan:.2%}")
print(f"Std Deviation: {sigma_tan:.2%}")
print(f"Sharpe Ratio: {sharpe_tan:.4f}")
print(f"Weights:")
tan_weights = pd.Series(w_tan, index=mu_hat.index).sort_values(ascending=False)
print(tan_weights.to_string())
print(f"Long positions: {(w_tan > 0).sum()}")
print(f"Short positions: {(w_tan < 0).sum()}")


Tangency Portfolio
Expected Return: 208.80%
Std Deviation: 44.02%
Sharpe Ratio: 4.7431
Weights:
GOOG     0.483914
ATO      0.415219
DTE      0.402636
LRCX     0.373313
REG      0.366821
WELL     0.352659
JPM      0.349019
ADI      0.341753
MS       0.335522
CAT      0.325608
AFL      0.304822
SO       0.304794
WRB      0.303594
BRK-B    0.271909
FI       0.269405
ED       0.260954
LLY      0.255699
DHI      0.253903
PM       0.246660
STLD     0.243056
PKG      0.236243
PGR      0.235132
AVY      0.227006
AEP      0.224823
MSFT     0.221126
AJG      0.218956
TPL      0.216331
BK       0.200176
PLD      0.199528
GRMN     0.198547
NWS      0.197091
RF       0.195524
DOV      0.195265
PHM      0.194780
KDP      0.192945
JNJ      0.190545
CTAS     0.190199
IEX      0.190088
BX       0.187878
CHD      0.183220
MAS      0.181589
PNC      0.180790
FITB     0.177580
TMO      0.177181
WEC      0.174912
ABBV     0.170294
OKE      0.167314
DE       0.166262
CBRE     0.165200
TMUS     0.163010
XOM

# Question 3.2

In [44]:
!pip install cvxpy

## Using cvxpy to solve the Std. Dev Minimization Problem (GMV)

In [70]:
import cvxpy as cp
n = len(mu_hat)
w = cp.Variable(n)
objective = cp.Minimize(cp.quad_form(w, sigma_hat))
constraints = [
    cp.sum(w) == 1
]
problem = cp.Problem(objective, constraints)
solution = problem.solve(solver=cp.OSQP, polish=True)

## Checking if the Solution matches

In [71]:
print(f"Std. Dev: {np.sqrt(solution)*100:.2f}%")

Std. Dev: 8.14%


## Adding Long-Only Constraint

In [72]:
n = len(mu_hat)
w = cp.Variable(n)
objective = cp.Minimize(cp.quad_form(w, sigma_hat))
constraints = [
    cp.sum(w) == 1,
    w >= 0
]
problem = cp.Problem(objective, constraints)
solution = problem.solve(solver=cp.OSQP, polish=True)

print(f"Std. Dev: {np.sqrt(solution)*100:.2f}%")

Std. Dev: 12.19%
